In [ ]:
### Import Setup ##

import sys
import os

#####################
# Import of utils.py functions
#####################
# Required to get utils.py and access its functions
repository_dir = os.path.abspath("")
sys.path.append(repository_dir)
sys.path.append('.')
from utils import loadFSL, FSLeyesServer, mkdir_no_exist


#############################
# Loading fsl and freesurfer within Neurodesk
#############################
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
await lmod.load('freesurfer/7.4.1')
await lmod.list()

####################
# Setup FSL path
####################
loadFSL()

###################
# Load all relevant libraries for the lab
##################
import fsl.wrappers
from fsl.wrappers import fslmaths


import os.path as op
import glob

import ants

# FSL function wrappers which we will call from python directly
from fsl.wrappers import fast, bet
from fsl.wrappers.misc import fslroi
from fsl.wrappers import flirt

# General purpose imports to handle paths, files etc
import glob
import pandas as pd
import numpy as np
import json
import subprocess

In [ ]:
%gui wx

In [ ]:
################
# Start FSLeyes 
################
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

In [ ]:
### Define Main Paths

base_root = repository_dir
deriv_root = op.join(base_root, 'derivatives')
preproc_root = op.join(base_root, 'derivatives','preprocessed_data')


### Create Necessary Folders for storing modified Images

mkdir_no_exist(deriv_root)
mkdir_no_exist(preproc_root)
mkdir_no_exist(op.join(preproc_root, 'anat'))
mkdir_no_exist(op.join(preproc_root, 'func'))
mkdir_no_exist(op.join(preproc_root, 'fmap'))

In [ ]:
### View T1 Image

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(base_root, 'T1','T1w.nii.gz'))

### Skull-Stripping

In [ ]:
### Define Paths

anatomical_path = op.join(base_root, 'T1', 'T1w.nii.gz')
betted_path = op.join(preproc_root, 'anat', 'T1w')

### Skull-Stripping

os.system('bet {} {} -m {}'.format(anatomical_path, betted_path, '-R' ))
print("Skull-Strpping with BET:: DONE")

resulting_mask_path = op.join(preproc_root,  'anat', 'T1w_mask')

In [ ]:
### View Mask Fit and Make Modifications (if needed)

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(base_root,'T1', 'T1w.nii.gz'))
fsleyesDisplay.load(resulting_mask_path)

In [ ]:
### Apply Mask

def apply_fsl_math_bet(img_path, mask_path, masked_img_path):

    os.system('fslmaths {} -mas {} {}'.format(img_path, mask_path, masked_img_path))

apply_fsl_math_bet(anatomical_path, betted_path, resulting_mask_path)

In [ ]:
### View Betted Brain

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(preproc_root, 'anat', 'T1w'))

### Preprocessing using fsl_anat

In [ ]:
### Define Paths

betted_path_grey = op.join(preproc_root, 'anat', 'T1w.nii.gz:colormap=greyscale')
mni_reference_path = op.expandvars('$FSLDIR/data/standard/MNI152_T1_1mm.nii.gz')

### View T1 alignment before Registration
import threading

def launch_freeview(img_list):
    """
    Wrapper around freeview to launch it with several images.
    This wrapper is necessary to launch freeview in a separate thread, ensuring the notebook is free to do something else.

    Parameters
    ----------
    img_list: list of string
        List of images (files) to load. Assumed by default to be volume files.
    """
    args = []
    
    for i in range(len(img_list)):
        args.append("-v")
        args.append(img_list[i])
    # Run the command
    subprocess.run(["freeview"] + args)

imgList = [mni_reference_path, 
           betted_path_grey] # Modify here this list to add any image you want, in .nii.gz format
freeview_thread = threading.Thread(target=launch_freeview, args=(imgList,)) # Remark the (imgList,) when passing to args. This is very important to make Thread work properly

# Start the thread
freeview_thread.start()

print("Freeview is running in a separate thread.")


In [ ]:
### Define Paths

anatomical_path = op.join(base_root, 'T1', 'T1w.nii.gz')

### Run FNIRT

import shutil

def fsl_anat_wrapped(anatomical_target, output_path):
    os.system('fsl_anat -i {} --clobber --nosubcortseg -o {}'.format(anatomical_target,output_path))
    # Now move all files from the output_path.anat folder created by FSL to 
    # the actual output_path
    fsl_anat_path = output_path+'.anat'
    files_to_move = glob.glob(op.join(fsl_anat_path, '*'))
    for f in files_to_move:
        shutil.move(f, op.join(output_path, op.split(f)[1]))
    
    # Remove the output_path.anat folder
    os.rmdir(fsl_anat_path)

fsl_anat_wrapped(anatomical_path, op.join(preproc_root, 'anat'))

In [ ]:
### View Segmentation Results

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(betted_path)
fsleyesDisplay.load(glob.glob(op.join(preproc_root, 'anat','*pve_0*'))[0])
fsleyesDisplay.load(glob.glob(op.join(preproc_root, 'anat','*pve_1*'))[0])
fsleyesDisplay.load(glob.glob(op.join(preproc_root, 'anat','*pve_2*'))[0])
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'

In [ ]:
### View Fit Registration

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm')))
fsleyesDisplay.load(op.join(preproc_root, 'anat', 'T1_to_MNI_lin'))
fsleyesDisplay.load(op.join(preproc_root, 'anat', 'T1_to_MNI_nonlin'))

In [ ]:
%%writefile .gitignore

T1/

fMRI/

derivatives/

.ipynb_checkpoints/

__pycache__/